In [0]:
# Definición de rutas base y constantes del catálogo analítico (Unity Catalog)
INPUT_PATH = "/Volumes/platform_dev/iol_challenge/input/*.csv"
tabla_destino = "platform_dev.bronce_byma.operaciones_raw"

# Configuración de Parámetros/Widgets para la orquestación centralizada.
# Permite alternar de forma homogénea entre reprocesamiento completo (1) o delta diario (0).
dbutils.widgets.text("full_load", "0")
full_load = int(dbutils.widgets.get("full_load"))
carga_full = (full_load == 1)

# Inicialización de Logging estándar para auditoría y trazabilidad en el clúster
import logging
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("ingesta_operaciones")

In [0]:
# COMMAND ----------

try:
    # Programación Defensiva: Si el entorno se limpia o se ejecuta desde cero,
    # el script auto-detecta la falta de la tabla y levanta el DDL de forma transparente.
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(
            f"La tabla {tabla_destino} no existe. "
            "Se ejecutará el notebook de creación de tablas."
        )
        # Ejecución interna del DDL físico resguardando esquemas
        dbutils.notebook.run("../DDL/creacion_tablas", 0)
        carga_full = True # Forzamos carga full para poblar la infraestructura naciente
        logger.info(
            "Tablas y esquemas creados correctamente. Se realizará una carga full."
        )

    elif carga_full:
        logger.warning(f"Carga full forzada manualmente para {tabla_destino}.")

    else:
        logger.info(f"La tabla {tabla_destino} existe.")
        
        # Validación de volumen: Si la tabla existe pero está vacía, se reconfigura a carga full
        # para evitar fallas lógicas en la lectura del Watermark temporal.
        tiene_datos = spark.sql(f"SELECT 1 FROM {tabla_destino} LIMIT 1").take(1) != []

        if tiene_datos:
            logger.info(f"La tabla {tabla_destino} contiene datos. Carga incremental activada.")
        else:
            carga_full = True
            logger.warning(
                f"La tabla {tabla_destino} está vacía. Se realizará una carga full."
            )

except Exception:
    logger.exception(f"Error crítico al validar o crear la tabla {tabla_destino}")
    raise

In [0]:
# COMMAND ----------

logger.info(f"Iniciando lectura del archivo: {INPUT_PATH}")

# Ingesta Raw: Se fuerza 'inferSchema=False' (todo String) para garantizar resiliencia en Bronce.
# Evita la pérdida o truncado de filas si producción cambia un tipo de dato sin previo aviso.
df = (
    spark.read
    .option("header", True)
    .option("delimiter", ",")
    .option("inferSchema", False) 
    .csv(INPUT_PATH)
    # Regionalización: BYMA opera en hora local. Convertimos el string base de UTC
    # al huso horario de Argentina para generar una 'fecha_particion' de negocio exacta.
    .withColumn(
        "fecha_particion",
        F.to_date(
            F.from_utc_timestamp(
                F.to_timestamp("fecha"),
                "America/Argentina/Buenos_Aires")))
    # Metadata Técnica: Timestamp del momento exacto de la ingesta en el Lakehouse
    .withColumn("_ingested_at", F.current_timestamp())
)

logger.info("Archivo leído correctamente y columnas técnicas generadas")

# Fase de Aislamiento Incremental (Corte Temporal)
if not carga_full:
    # Recuperación segura del Watermark temporal
    fila_control = spark.sql(f"""
        SELECT ultima_fecha_procesada
        FROM platform_dev.bronce_byma.control_cargas
        WHERE proceso = '{tabla_destino}'
    """).first()

    # Protección contra Cold Start: Si la tabla de control no tiene registros para este proceso,
    # capturamos defensivamente el NoneType y seteamos una fecha base para no romper el flujo.
    if fila_control is not None and fila_control["ultima_fecha_procesada"] is not None:
        ultima_fecha = fila_control["ultima_fecha_procesada"]
        logger.info(f"Carga incremental para {tabla_destino}. Última fecha procesada: {ultima_fecha}")
    else:
        ultima_fecha = "1970-01-01"
        logger.warning(
            f"No se encontró registro de control para {tabla_destino} (Arranque en frío). "
            f"Se procesará desde la fecha base: {ultima_fecha}"
        )

    # Filtrado estricto por la frontera de datos ya procesados
    df = df.filter(F.col("fecha_particion") > F.lit(ultima_fecha))
    logger.info(f"Filtro aplicado: fecha_particion > {ultima_fecha}")

else:
    logger.info(f"Carga full para {tabla_destino}. No se aplica filtro por fecha.")

In [0]:
# COMMAND ----------

cantidad_registros = df.count()

try:
    if carga_full:
        # Modo Destructivo Seguro: Borra el histórico y recrea los archivos físicos (Idempotencia)
        logger.info(f"Carga full: iniciando escritura de {cantidad_registros} registros en {tabla_destino}")
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(tabla_destino)
        logger.info(f"Carga full finalizada: {cantidad_registros} registros insertados")

    else:
        # Modo Incremental: Acumula las nuevas particiones sin alterar el pasado operativo
        if cantidad_registros == 0:
            logger.info("Carga incremental: no hay registros nuevos para insertar")
        else:
            logger.info(f"Carga incremental: iniciando escritura de {cantidad_registros} registros en {tabla_destino}")
            df.write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(tabla_destino)
            logger.info(f"Carga incremental finalizada: {cantidad_registros} registros insertados")

except Exception:
    logger.exception(f"Error durante la escritura en {tabla_destino}")
    raise

In [0]:
# Buscamos el valor máximo del lote procesado para avanzar la ventana en el próximo ciclo
ultima_fecha_cargada = df.agg(F.max("fecha_particion").alias("ultima_fecha")).first()["ultima_fecha"]

if ultima_fecha_cargada is not None:
    try:
        logger.info(f"Actualizando watermark del proceso {tabla_destino} con fecha {ultima_fecha_cargada}")

        # Uso de MERGE para garantizar atomicidad e idempotencia en la matriz de control de cargas
        spark.sql(f"""
        MERGE INTO platform_dev.bronce_byma.control_cargas AS target
        USING (
            SELECT
                '{tabla_destino}' AS proceso,
                DATE('{ultima_fecha_cargada}') AS ultima_fecha_procesada,
                current_timestamp() AS fecha_actualizacion
        ) AS source
        ON target.proceso = source.proceso

        WHEN MATCHED THEN UPDATE SET
            target.ultima_fecha_procesada = source.ultima_fecha_procesada,
            target.fecha_actualizacion = source.fecha_actualizacion

        WHEN NOT MATCHED THEN INSERT (proceso, ultima_fecha_procesada, fecha_actualizacion)
        VALUES (source.proceso, source.ultima_fecha_procesada, source.fecha_actualizacion)
        """)

        logger.info(f"Watermark actualizado correctamente: proceso={tabla_destino}, fecha={ultima_fecha_cargada}")

    except Exception:
        logger.exception(f"Error al actualizar watermark para {tabla_destino}")
        raise
else:
    logger.warning("No se actualizó la watermark porque no había registros nuevos procesados.")

In [0]:
# Auditoría de Calidad de Datos (DQ Gate)
# Proceso desacoplado para identificar colisiones de IDs transaccionales en Bronce.
# Permite monitorear la salud operacional de las fuentes sin frenar el flujo principal de carga.
df_duplicados = (
    spark.table("platform_dev.bronce_byma.operaciones_raw")
    .groupBy("id_transaccion")
    .agg(F.count("*").alias("count_dup"))
    .filter(F.col("count_dup") > 1)
    .withColumn("fecha_deteccion", F.current_timestamp())
)

# Guardamos el histórico de duplicados como registro analítico de calidad
df_duplicados.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("platform_dev.bronce_byma.calidad_duplicados")